# Demo of MNIST inference live on webcam-feed


In [ ]:
bitfile_path = 'bitfile_axi_master/'
#bitfile_path = 'bitfile_axi_stream/'

shape_img = (1, 28, 28, 1) # 1 frame, 28 width, 28 height, 1 channel
shape_y = (1, 10) # 1 frame, digits from 0 to 9 

Load the model to the FPGA fabric (PL)

In [6]:
import numpy as np
# import the library/driver which is common for every VItis Unified synthesis
from bitfile_axi_master.axi_master_driver import NeuralNetworkOverlay
#from bitfile_axi_stream.axi_stream_driver import NeuralNetworkOverlay

# create the overlay object
overlay = NeuralNetworkOverlay(bitfile_name=bitfile_path, x_shape=shape_img, y_shape=shape_y, dtype=np.float32)


ModuleNotFoundError: No module named 'pynq'

In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert full frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Resize full frame to 28x28
    resized = cv2.resize(gray, (28, 28))

    # Threshold for cleaner digit (better than simple invert)
    _, resized = cv2.threshold(resized, 120, 255, cv2.THRESH_BINARY_INV)

    # Normalize
    resized = resized.astype("float32") / 255.0

    # Reshape to (1,28,28,1)
    input_img = resized.reshape(1, 28, 28, 1)

    # Predict
    prediction = model.predict(input_img, verbose=0)
    # Do the prediction/run inference
    result = overlay.predict(frame, debug=False, profile=True, encode=np.float32, decode=np.float32)
    pred = result[0]
    pred_rate = result[1]
    digit = np.argmax(pred)
    confidence = np.max(pred)

    # Display prediction
    cv2.putText(frame, f"Prediction: {digit} ({confidence:.2f})",
                (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,0,255), 2)

    cv2.imshow("Live MNIST Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


....